# Auditing a recidivism model with `c4fairness` (COMPAS)

A model that predicts whether a defendant will re-offend can be accurate overall yet wrong in
*different ways* for different groups. The COMPAS controversy centred on exactly this: the
tool's **false-positive rate** — labelling someone high-risk who did **not** re-offend — was
higher for Black defendants.

Group-level rates (FPR per race) are the standard lens. `c4fairness` adds a second one: it
**clusters the test set** on the features and reports the error rate *per cluster*, then shows
which demographic groups each cluster concentrates. Clusters can expose pockets of high error
that a single per-race number averages away.

This notebook runs that audit end to end on `Data/compas/Compas_error_shap.csv` (5050 rows,
each a defendant with the model's prediction), using the Python API.

In [ ]:
import numpy as np
import pandas as pd
from c4fairness.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

raw = pd.read_csv("../Data/compas/Compas_error_shap.csv")
# The file also ships pre-computed scaled / one-hot / SHAP columns; keep the readable ones so
# the pipeline can encode `race`/`sex` itself without colliding with existing `race_*` columns.
df = raw[["age", "priors_count", "sex", "race", "true_class", "predicted_class"]].copy()
print(df.shape)
df.head()

## 1. The columns

- `true_class`, `predicted_class` — ground truth and the model's 0/1 output. `1` = predicted
  to re-offend. The error we audit is derived from these two.
- `age`, `priors_count` — the features we cluster on (`regular`).
- `sex` (binary), `race` (6 categories), `age` (numeric) — the protected attributes
  (`sensitive`) we check each cluster against.

`age` is both a clustering feature and a sensitive attribute — that's fine; the two roles are
independent.

In [ ]:
# The confusion matrix the whole audit rests on (positive class = 1 = "will re-offend").
tp = ((df.true_class == 1) & (df.predicted_class == 1)).sum()
fp = ((df.true_class == 0) & (df.predicted_class == 1)).sum()
fn = ((df.true_class == 1) & (df.predicted_class == 0)).sum()
tn = ((df.true_class == 0) & (df.predicted_class == 0)).sum()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"overall FPR = FP/(FP+TN) = {fp/(fp+tn):.3f}")

## 2. The aggregate view (what clustering will refine)

FPR is a *conditional* rate: it is defined only over the actual non-re-offenders
(`true_class == 0`). Below is the per-race FPR — the standard group-fairness number.

In [ ]:
neg = df[df.true_class == 0]           # FPR's denominator = actual negatives
per_race_fpr = (neg.assign(fp=(neg.predicted_class == 1))
                   .groupby("race")["fp"].mean().sort_values(ascending=False))
per_race_fpr.round(3)

That's the flat, one-number-per-group summary. Now we let clustering find structure *within*
the feature space and report FPR per cluster.

## 3. Encode + cluster

`encode_categoricals` one-hot-encodes the string columns (`sex`, `race`) for the Euclidean
distance and returns `multiclass_dummies` (mcd) — the mapping from `race` to its dummy columns,
which we use later to rebuild a readable `race` column for the tables. The 0/1 dummies are kept
out of `StandardScaler` (scaling binary indicators distorts distances).

In [ ]:
from c4fairness.preprocessing import encode_categoricals
from c4fairness.clustering import cluster

sensitive = ["sex", "race", "age"]
col_lists = {"regular": ["age", "priors_count"], "sensitive": sensitive, "proxy": [], "special": []}
orig_sensitive = list(sensitive)

dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, [], "kmeans", distance="euclidean"
)
print("race ->", mcd["race"])           # the one-hot columns race was expanded into
clustering_cols = cl["regular"] + cl["sensitive"]
clustering_cols

In [ ]:
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=4, random_state=42)
print(f"k = {res.n_clusters}   silhouette = {res.silhouette:.3f}")
print("cluster sizes:", res.cluster_sizes)

## 4. Define the error: false-positive rate

`binary_error_rate_column` turns `true_class`/`predicted_class` into a per-row column whose
**mean over any set of rows is that set's FPR**. The trick: rows outside the rate's denominator
(the actual *positives*, which FPR ignores) are set to `NaN`, so every downstream metric that
averages the column — and every 2×2 significance test — reads the conditional rate for free.

In [ ]:
from c4fairness.fairness_metrics import binary_error_rate_column

fpr_col = binary_error_rate_column(df["true_class"], df["predicted_class"], "fpr")
print(fpr_col.value_counts(dropna=False))   # 1 = FP, 0 = TN, NaN = actual positive (excluded)
print("mean over non-NaN =", round(fpr_col.mean(), 3), "= overall FPR")

## 5. Build the per-cluster recap

`make_recap` produces one row per cluster. We show sensitive features in **salient** form (one
readable column per feature with its dominant category), which needs two steps: build the
analysis column list, and rebuild the readable `race` column from its dummies into the recap
frame (`apply_salient_reconstruction`). `age` is declared continuous so it's summarised by its
median rather than treated as categories.

In [ ]:
from c4fairness.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4fairness.experiments import make_recap

analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")

dfe["fpr"] = fpr_col.values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)   # rebuild readable 'race'

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="fpr", error_type="binary",
                   feature_matrix=res.feature_matrix, continuous_sensitive_cols=["age"])
recap.round(3)

## 6. Reading the recap

Per cluster:

- **`error_value`** — the cluster's FPR.
- **`error_gap`** — FPR of this cluster minus the FPR of all other rows (one-vs-all); positive
  = worse than the rest.
- **`error_gap_sig`** — Fisher-exact p-value for that gap (is this cluster's FPR really
  different?). Lower = more significant.
- **`race_value` / `race_cat`** — the proportion in the dominant race and which race that is.
- **`race_gap_sig`** — Chi-square p-value for whether race composition differs across clusters.
- **`sex_Male_value`**, **`age_value`** — male proportion and median age.

Sort by FPR to find the disparate cluster and see who it concentrates:

In [ ]:
view = recap[["c", "count", "error_value", "error_gap", "error_gap_sig",
              "race_cat", "sex_Male_value", "age_value"]].sort_values("error_value", ascending=False)
view.round(3)

In [ ]:

worst = int(view.iloc[0]["c"])
inside = res_df[res_df["clusters"] == worst]
print(f"cluster {worst}: n={len(inside)}, FPR={recap.loc[recap.c==worst,'error_value'].iloc[0]:.3f}")
inside["race"].value_counts(normalize=True).round(3)

## 7. Heatmap

The heatmap encodes the same table by colour family — blue = size, red = error, violet =
sensitive; p-value columns are darker when more significant. The `error_label` renames the
error column, and `sensitive_labels` renames features for display.

In [ ]:


plot_cluster_recap_heatmap(recap.copy(), "compas_fpr", ".", error_label="FP Rate",
                           sensitive_labels={"race": "Ethnicity"})
Image("compas_fpr.png")

## Takeaway & variations

The cluster with the highest FPR and a significant `FP Rate gap sig.` is where the model most
over-predicts risk; its `race`/`age` columns show which defendants land there — a disparity
localised to a subgroup, not just an overall per-race average.

- **Audit the other direction:** swap `"fpr"` for `"fnr"` (missed re-offenders).
- **Let k vary:** pass `n_min`/`n_max` to `cluster` for silhouette-based k selection.
- **Batch it:** the CLI `--experiment` mode runs every feature-group combination and writes an
  Overview table + heatmaps comparing them.